In [1]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

Python: 3.13.16 (main, Oct  1 2026, 20:59:40) [MSC v.1944 64 bit (AMD64)]
Executable: c:\Users\Gabri\Documents\GitHub\Programacion_Para_IA\programacion_ai\amazon-products\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from support.data import load, catalog_text, prepare_review_texts

reviews, products = load(Path("datasets"))

product_ids = list(products)

catalog_texts = [
    catalog_text(products[product_id])
    for product_id in product_ids
]

texts, review_ids, review_product_ids = prepare_review_texts(reviews)

In [3]:
print("Products:", len(product_ids))
print("Catalog texts:", len(catalog_texts))
print("Total reviews:", len(reviews))
print("Reviews with text:", len(texts))
print("Review IDs:", len(review_ids))
print("Review product IDs:", len(review_product_ids))

Products: 600
Catalog texts: 600
Total reviews: 6992
Reviews with text: 6992
Review IDs: 6992
Review product IDs: 6992


In [4]:
print("Product ID:", product_ids[0])
print("Catalog text:", catalog_texts[0][:200])

Product ID: B0000BYDKO
Catalog text: Cord Storage Reel w/Stand Handle driven cord storage reel Holds up to 150ft of 16/3 cord Easy carry handle Wide base for stability Floor standing Product Description The Cord Storage Reel is designed 


In [5]:
from support.settings import load_settings
from support.encoder import load_model, encode_texts

settings = load_settings(Path("config.json"))
model = load_model(settings)

catalog_embeddings = encode_texts(
    model,
    catalog_texts,
    settings.batch_size,
)

c:\Users\Gabri\Documents\GitHub\Programacion_Para_IA\programacion_ai\amazon-products\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7671.78it/s]


In [6]:
print("Shape:", catalog_embeddings.shape)
print("First vector:", catalog_embeddings[0].shape)
print("First 5 norms:", catalog_embeddings.norm(dim=1)[:5])
print("Product IDs:", len(product_ids))

Shape: torch.Size([600, 384])
First vector: torch.Size([384])
First 5 norms: tensor([1., 1., 1., 1., 1.])
Product IDs: 600


In [7]:
review_embeddings = encode_texts(
    model,
    texts,
    settings.batch_size,
)

In [8]:
print("Review embeddings:", review_embeddings.shape)
print("First review vector:", review_embeddings[0].shape)
print("First 5 norms:", review_embeddings.norm(dim=1)[:5])
print("Review IDs:", len(review_ids))
print("Review product IDs:", len(review_product_ids))

Review embeddings: torch.Size([6992, 384])
First review vector: torch.Size([384])
First 5 norms: tensor([1., 1., 1., 1., 1.])
Review IDs: 6992
Review product IDs: 6992


In [9]:
first_product_id = product_ids[0]

review_positions = [
    i
    for i, product_id in enumerate(review_product_ids)
    if product_id == first_product_id
]

print("Product ID:", first_product_id)
print("Number of reviews:", len(review_positions))
print("First positions:", review_positions[:5])

Product ID: B0000BYDKO
Number of reviews: 30
First positions: [0, 1, 2, 3, 4]


In [10]:
product_review_vectors = review_embeddings[review_positions]

review_average = product_review_vectors.mean(dim=0)

print("Selected vectors:", product_review_vectors.shape)
print("Average vector:", review_average.shape)
print("Average norm:", review_average.norm())

Selected vectors: torch.Size([30, 384])
Average vector: torch.Size([384])
Average norm: tensor(0.4765)


In [11]:
import torch.nn.functional as F

review_summary = F.normalize(review_average, dim=0)

print("Original norm:", review_average.norm())
print("Normalized norm:", review_summary.norm())
print("Summary shape:", review_summary.shape)

Original norm: tensor(0.4765)
Normalized norm: tensor(1.)
Summary shape: torch.Size([384])


In [12]:
review_summaries = []

for product_id in product_ids:
    positions = [
        i
        for i, review_product_id in enumerate(review_product_ids)
        if review_product_id == product_id
    ]

    product_vectors = review_embeddings[positions]
    average = product_vectors.mean(dim=0)
    normalized_average = F.normalize(average, dim=0)

    review_summaries.append(normalized_average)

In [13]:
import torch

review_summary_matrix = torch.stack(review_summaries)

print("Number of summaries:", len(review_summaries))
print("Matrix shape:", review_summary_matrix.shape)
print("First 5 norms:", review_summary_matrix.norm(dim=1)[:5])

Number of summaries: 600
Matrix shape: torch.Size([600, 384])
First 5 norms: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [14]:
from support.encoder import summarize_reviews

new_review_summaries = summarize_reviews(
    product_ids,
    review_product_ids,
    review_embeddings,
)

print("Shape:", new_review_summaries.shape)
print("First 5 norms:", new_review_summaries.norm(dim=1)[:5])

Shape: torch.Size([600, 384])
First 5 norms: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [15]:
print(
    "Same results:",
    torch.allclose(new_review_summaries, review_summary_matrix),
)

Same results: True


In [16]:
catalog_vector = catalog_embeddings[0]
reviews_vector = new_review_summaries[0]

combined_vector = 0.5 * catalog_vector + 0.5 * reviews_vector

print("Catalog shape:", catalog_vector.shape)
print("Reviews shape:", reviews_vector.shape)
print("Combined shape:", combined_vector.shape)

print("Catalog norm:", catalog_vector.norm())
print("Reviews norm:", reviews_vector.norm())
print("Combined norm:", combined_vector.norm())

Catalog shape: torch.Size([384])
Reviews shape: torch.Size([384])
Combined shape: torch.Size([384])
Catalog norm: tensor(1.)
Reviews norm: tensor(1.)
Combined norm: tensor(0.8594)


In [17]:
final_vector = F.normalize(combined_vector, dim=0)

print("Product ID:", product_ids[0])
print("Final vector shape:", final_vector.shape)
print("Final vector norm:", final_vector.norm())

Product ID: B0000BYDKO
Final vector shape: torch.Size([384])
Final vector norm: tensor(1.)


In [18]:
combined_matrix = (
    0.5 * catalog_embeddings
    + 0.5 * new_review_summaries
)

final_matrix = F.normalize(combined_matrix, dim=1)

print("Combined matrix:", combined_matrix.shape)
print("Final matrix:", final_matrix.shape)
print("Product IDs:", len(product_ids))
print("First 5 norms:", final_matrix.norm(dim=1)[:5])

Combined matrix: torch.Size([600, 384])
Final matrix: torch.Size([600, 384])
Product IDs: 600
First 5 norms: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [19]:
print(
    "First vector matches:",
    torch.allclose(final_matrix[0], final_vector),
)

print(
    "All vectors normalized:",
    torch.allclose(
        final_matrix.norm(dim=1),
        torch.ones(600),
    ),
)

First vector matches: True
All vectors normalized: True


In [20]:
import importlib
import support.encoder as encoder

importlib.reload(encoder)

verified_matrix = encoder.combine_embeddings(
    catalog_embeddings,
    new_review_summaries,
)

print("Matrix shape:", verified_matrix.shape)
print(
    "Matches manual calculation:",
    torch.allclose(verified_matrix, final_matrix),
)
print(
    "All vectors normalized:",
    torch.allclose(
        verified_matrix.norm(dim=1),
        torch.ones(len(product_ids)),
    ),
)

Matrix shape: torch.Size([600, 384])
Matches manual calculation: True
All vectors normalized: True


In [21]:
query = "Comfortable headphones with good sound"

query_vectors = model.encode(
    [query],
    batch_size=settings.batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

query_vector = query_vectors[0]

print("Query matrix:", query_vectors.shape)
print("Query vector:", query_vector.shape)
print("Query norm:", query_vector.norm())

Query matrix: torch.Size([1, 384])
Query vector: torch.Size([384])
Query norm: tensor(1.)


In [22]:
importlib.reload(encoder)

verified_query = encoder.encode_query(
    model,
    query,
    settings.batch_size,
)

print("Query shape:", verified_query.shape)
print("Query norm:", verified_query.norm())
print(
    "Matches manual calculation:",
    torch.allclose(verified_query, query_vector),
)

Query shape: torch.Size([384])
Query norm: tensor(1.)
Matches manual calculation: True


In [23]:
print("Products:", len(product_ids))
print("Unique product IDs:", len(set(product_ids)))

print("Catalog rows:", catalog_embeddings.shape[0])
print("Review summary rows:", new_review_summaries.shape[0])
print("Final matrix rows:", verified_matrix.shape[0])

print("Product IDs match dictionary:", product_ids == list(products))

print(
    "All review IDs belong to known products:",
    set(review_product_ids).issubset(products),
)

Products: 600
Unique product IDs: 600
Catalog rows: 600
Review summary rows: 600
Final matrix rows: 600
Product IDs match dictionary: True
All review IDs belong to known products: True


In [24]:
print("Catalog finite:", torch.isfinite(catalog_embeddings).all().item())
print("Reviews finite:", torch.isfinite(review_embeddings).all().item())
print("Summaries finite:", torch.isfinite(new_review_summaries).all().item())
print("Final matrix finite:", torch.isfinite(verified_matrix).all().item())

Catalog finite: True
Reviews finite: True
Summaries finite: True
Final matrix finite: True


In [25]:
for name, matrix in [
    ("Catalog", catalog_embeddings),
    ("Review summaries", new_review_summaries),
    ("Final matrix", verified_matrix),
]:
    norms = matrix.norm(dim=1)

    print(name)
    print("  Min norm:", norms.min().item())
    print("  Max norm:", norms.max().item())
    print("  All normalized:", torch.allclose(norms, torch.ones_like(norms)))

Catalog
  Min norm: 0.9999998807907104
  Max norm: 1.0000001192092896
  All normalized: True
Review summaries
  Min norm: 0.9999998211860657
  Max norm: 1.0000001192092896
  All normalized: True
Final matrix
  Min norm: 0.9999998807907104
  Max norm: 1.0000001192092896
  All normalized: True


In [26]:
try:
    encoder.summarize_reviews(
        ["UNKNOWN_PRODUCT"],
        review_product_ids,
        review_embeddings,
    )
except ValueError as error:
    print("Expected error:", error)

Expected error: No review embeddings for product UNKNOWN_PRODUCT
